In [1]:
%pip install -U langsmith openevals requests python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.4/99.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 805.0/805.0 kB 21.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.0/107.0 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.9/125.9 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.0/572.0 kB 38.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langsmith
    Found existing installation: langsmith 0.12.1
    Uninstalling langsmith-0.12.1:
      Successfully uninstalled langsmith-0.12.1
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.1
    Uninstalling langchain-core-1.6.1:
      Successfully uninstalled langchain-core-1.6.1

In [1]:
import os

print(os.getcwd())

c:\Users\asus\Rawaj_Project_SDA\agents\research_agent\evaluation


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

print("LangSmith key:", bool(os.getenv("LANGSMITH_API_KEY")))
print("OpenAI key:", bool(os.getenv("OPENAI_API_KEY")))

LangSmith key: True
OpenAI key: True


In [3]:
%pip install -U  openevals  

  Using cached openevals-0.2.0-py3-none-any.whl.metadata (99 kB)
  Using cached langchain-1.4.2-py3-none-any.whl.metadata (6.2 kB)
Using cached openevals-0.2.0-py3-none-any.whl (106 kB)
Using cached langchain-1.4.2-py3-none-any.whl (163 kB)
   ---------------------------------------- 0.0/572.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/572.0 kB ? eta -:--:--
   ------------------ --------------------- 262.1/572.0 kB ? eta -:--:--
   ---------------------------------------- 572.0/572.0 kB 2.8 MB/s  0:00:00

  Attempting uninstall: langchain-core

    Found existing installation: langchain-core 1.6.2

    Uninstalling langchain-core-1.6.2:

      Successfully uninstalled langchain-core-1.6.2

   ---------------------------------------- 0/3 [langchain-core]
   ---------------------------------------- 0/3 [langchain-core]
   ---------------------------------------- 0/3 [langchain-core]
   ---------------------------------------- 0/3 [langchain-core]
   ---------------

In [5]:
import json
import base64
import requests
from pathlib import Path

from langsmith import Client
from openevals.llm import create_llm_as_judge


import sys

project_root = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "agents" / "research_agent" / "research_analyzer.py").is_file()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError("Could not locate the project root.")

sys.path.insert(0, str(project_root))

from agents.research_agent.research_analyzer import _analyze_single_content

JSON_PATH = project_root / "results" / "dearduck_sa_research_20260922_112749.json"

with open(JSON_PATH, "r", encoding="utf-8") as f:
    report = json.load(f)

print("Restaurant:", report["restaurant"]["name"])
print("Number of content items:", len(report["content"]))

Restaurant: Dear Duck
Number of content items: 20


In [6]:
item = report["content"][0]

print("CONTENT ID:")
print(item["content_id"])

print("\nCAPTION:")
print(item["raw_data"]["caption"])

print("\nCURRENT ANALYSIS:")
print(
    json.dumps(
        item["analysis"],
        indent=2,
        ensure_ascii=False
    )
)

CONTENT ID:
3991129199864779860

CAPTION:
Don’t miss out on our Saudi National Day offers!🇸🇦🐣

New Menu, New Exciting Combos! 
Book Your Spot Today ✨

Everyday from 8AM-11PM💛

#اليومالوطنيالسعودي #dearduck # jeddah #عروض #lunch

CURRENT ANALYSIS:
{
  "content_type": "Reel with cover image and transcript",
  "content_categories": [
    "national_day_campaign",
    "food_combos",
    "restaurant_offer"
  ],
  "content_pillar": "offers_campaigns",
  "content_themes": [
    "Saudi National Day",
    "Signature shakshouka-ish",
    "Chicken and Waffle",
    "96 SAR combo"
  ],
  "cta": {
    "present": true,
    "intent": "book a table",
    "description": "The caption asks viewers to book their spot today."
  },
  "promotion": {
    "present": true,
    "intent": "Saudi National Day combo offer",
    "description": "The post promotes Saudi National Day combos featuring Signature shakshouka-ish and Chicken and Waffle for 96 SAR."
  },
  "visual_signals": {
    "food_visible": true,
    "men

LangSmith Dataset

In [7]:
client = Client()

DATASET_NAME = "rawaj-research-agent-judge-v1"

In [8]:
examples = []

for item in report["content"]:

    content_input = {
        "content_id": item.get("content_id"),
        "short_code": item.get("short_code"),

        "raw_data": item.get("raw_data"),

        "reel_details": item.get("reel_details"),
    }

    examples.append(
        {
            "inputs": {
                "content": content_input
            },

            "metadata": {
                "restaurant": report["restaurant"]["name"],
                "content_id": item.get("content_id"),
                "short_code": item.get("short_code"),
            },
        }
    )

print("Examples prepared:", len(examples))

Examples prepared: 20


Testing 3 Posts

In [9]:
test_examples = examples[:3]

len(test_examples)

3

Run this cell only once

In [10]:
dataset = client.create_dataset(
    dataset_name=DATASET_NAME,
    description=(
        "Raw Instagram evidence used to evaluate "
        "the Rawaj Research Agent using LLM-as-a-judge."
    ),
)

client.create_examples(
    dataset_id=dataset.id,
    examples=test_examples,
)

print("Dataset created:", DATASET_NAME)

Dataset created: rawaj-research-agent-judge-v1


In [12]:
JUDGE_PROMPT = """
You are evaluating Rawaj's Instagram Research Agent.

The INPUT contains the original Instagram evidence available
to the Research Agent.

The OUTPUT contains the analysis produced by the Research Agent.

You must determine whether the analysis is accurate, grounded
in the supplied evidence, and appropriate for a Research Agent.

Evaluate the following:

1. EVIDENCE GROUNDING
Every factual claim should be supported by the caption,
transcript, metadata, or supplied image evidence.

2. VISUAL ACCURACY
Check whether claims such as:
- food_visible
- menu_visible
- price_visible
- offer_visible
- logo_visible
- branding_visible
- people_visible
- text_in_visual
- visual_summary

are consistent with the supplied images.

3. CONTENT INTERPRETATION
Check whether:
- content_pillar
- content_categories
- content_themes
- CTA
- promotion

are reasonable interpretations of the evidence.

Do not require exact wording when two labels have essentially
the same meaning.

4. SUMMARY FAITHFULNESS
caption_summary, visual_summary and transcript_summary
must not invent information.

5. EVIDENCE QUALITY
Evidence items should genuinely support the fields they claim
to support.

6. RESEARCH ROLE COMPLIANCE
The Research Agent may observe, describe and analyze evidence.

It must NOT:
- qualify the restaurant
- decide whether it is a good or bad prospect
- assign a prospect score
- recommend marketing improvements
- identify marketing gaps
- create a marketing strategy

Return PASS when the analysis is materially accurate,
grounded and stays within the Research Agent role.

Return FAIL when there is a meaningful hallucination,
unsupported claim, incorrect interpretation, visual error,
or role violation.

Briefly explain your decision.

<input>
{inputs}
</input>

<output>
{outputs}
</output>

<images>
{attachments}
</images>
"""

In [13]:
judge = create_llm_as_judge(
    prompt=JUDGE_PROMPT,
    model="gpt-5.6-terra",
    feedback_key="research_quality",
)

Image conversion helper

In [20]:
import base64
import requests


def image_url_to_attachment(url: str):
    try:
        response = requests.get(
            url,
            timeout=30,
            headers={
                "User-Agent": "Mozilla/5.0"
            }
        )

        response.raise_for_status()

        mime_type = (
            response.headers
            .get("Content-Type", "image/jpeg")
            .split(";")[0]
            .lower()
        )

        # Only formats supported by OpenEvals/OpenAI vision
        supported_types = {
            "image/jpeg",
            "image/png",
            "image/gif",
            "image/webp",
        }

        if mime_type not in supported_types:
            print(
                f"⚠️ Unsupported response type: {mime_type}"
            )
            return None

        encoded = base64.b64encode(
            response.content
        ).decode("utf-8")

        # IMPORTANT:
        # This must be a complete data URI.
        data_uri = (
            f"data:{mime_type};base64,{encoded}"
        )

        return {
            "mime_type": mime_type,
            "data": data_uri,
        }

    except Exception as e:
        print(f"⚠️ Could not load image: {e}")
        return None

In [24]:
#image loading test

first_image_url = (
    report["content"][0]
    ["raw_data"]
    ["image_urls"][0]
)

attachment = image_url_to_attachment(
    first_image_url
)

print("Loaded:", attachment is not None)

if attachment:
    print("Mime type:", attachment["mime_type"])
    print(
        "Beginning:",
        attachment["data"][:50]
    )


Loaded: True
Mime type: image/jpeg
Beginning: data:image/jpeg;base64,/9j/4AAQSkZJRgABAQAAAQABAAD


In [22]:
#LLM-as-a-judge evaluation function
def research_quality_evaluator(
    inputs: dict,
    outputs: dict,
    reference_outputs=None,
):

    content = inputs["content"]

    raw_data = content.get(
        "raw_data",
        {}
    )

    image_urls = raw_data.get(
        "image_urls",
        []
    )

    attachments = []

    for url in image_urls:

        attachment = image_url_to_attachment(
            url
        )

        if attachment is not None:
            attachments.append(
                attachment
            )

    result = judge(
        inputs=inputs,
        outputs=outputs,
        attachments=attachments,
    )

    return result

In [23]:
#target function for evaluation
def target(inputs: dict) -> dict:

    content = inputs["content"]

    result = _analyze_single_content(
        content
    )

    if hasattr(result, "model_dump"):
        result = result.model_dump()

    return {
        "analysis": result
    }

In [18]:
#test the target function with the first example
test_input = test_examples[0]["inputs"]

result = target(test_input)

print(
    json.dumps(
        result,
        indent=2,
        ensure_ascii=False
    )
)

{
  "analysis": {
    "content_id": "3991129199864779860",
    "short_code": "DdjVtyCoRRU",
    "raw_data": {
      "content_url": "https://www.instagram.com/p/DdjVtyCoRRU/",
      "timestamp": "2026-09-21T14:11:04.000Z",
      "caption": "Don’t miss out on our Saudi National Day offers!🇸🇦🐣\n\nNew Menu, New Exciting Combos! \nBook Your Spot Today ✨\n\nEveryday from 8AM-11PM💛\n\n#اليومالوطنيالسعودي #dearduck # jeddah #عروض #lunch",
      "content_type": "Video",
      "product_type": "clips",
      "image_urls": [
        "https://instagram.fpoa5-1.fna.fbcdn.net/v/t51.71878-15/818492181_1468796638443071_8238676509362947128_n.jpg?stp=dst-jpg_e15_tt6&_nc_cat=101&_nc_map=urlgen_bucketless&ig_cache_key=Mzk5MTEyOTE5OTg2NDc3OTg2MDEzNzE3NjA0MDQ5NDEyMzc%3D.3-ccb7-5&ccb=7-5&_nc_sid=58cdad&efg=eyJ2ZW5jb2RlX3RhZyI6IkNMSVBTLnhwaWRzLjY0MC5zZHIudmlkZW9fbmZyYW1lX2NvdmVyX2ZyYW1lLkMzIn0%3D&_nc_ohc=wq4C8FzSLH8Q7kNvwGEv3KX&_nc_oc=AdpO3QYLI7EEdoTUGvsJh_gtEwNjZGbyyyDq_kajCiNE7irXRvqml-5YIFoRfzGafhU&_nc_ad=z

In [25]:
judge_result = research_quality_evaluator(
    inputs=test_input,
    outputs=result,
)

judge_result

{'key': 'research_quality',
 'score': True,
 'comment': 'The analysis is grounded in the caption and supplied cover image. The visual text, two food images, Dear Duck branding/duck logo, Saudi National Day combo offer, and 96 SAR price are all visible. The caption supports the booking CTA, new combos, promotion, and daily hours. The transcript is accurately characterized as music, and the response stays within an observational research role without prospect qualification or recommendations. Thus, the score should be: true.',
 'metadata': None}

In [26]:
results = client.evaluate(
    target,
    data=DATASET_NAME,

    evaluators=[
        research_quality_evaluator
    ],

    experiment_prefix="rawaj-research-agent-judge-test",

    description=(
        "Multimodal LLM-as-a-judge evaluation "
        "of Rawaj Research Agent."
    ),

    max_concurrency=0,
)

c:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'rawaj-research-agent-judge-test-9e6fe6c5' at:
https://smith.langchain.com/o/49aa18a5-8e0b-4057-b558-c6df275d284d/datasets/c3fc7d12-eb2d-4c91-9017-770e0c4b2f78/compare?selectedSessions=6d236b1c-e1a6-4996-832b-b712759c8b90




3it [00:48, 16.02s/it]
